# Leukemia Detection using VGG16 Transfer Learning

Binary classification: **ALL (Leukemia)** vs **HEM (Healthy)**

## Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow:', tf.__version__)

## Dataset Loading

In [ ]:
BASE_DIR = '/kaggle/input/leukemia/'

# List available folds
for folder in sorted(os.listdir(BASE_DIR)):
    path = os.path.join(BASE_DIR, folder)
    if os.path.isdir(path):
        print(folder, '->', os.listdir(path))

In [ ]:
import glob, shutil

# Merge fold_0 + fold_1 -> train, fold_2 -> val
train_dir = '/tmp/data/train'
val_dir   = '/tmp/data/val'

# Clean if exists
if os.path.exists('/tmp/data'):
    shutil.rmtree('/tmp/data')

folds = sorted(glob.glob(os.path.join(BASE_DIR, 'fold_*')))

for cls in ['all', 'hem']:
    # Train: fold_0 + fold_1
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    for fold in folds:
        if 'fold_2' in fold:
            continue
        src = os.path.join(fold, cls)
        for f in os.listdir(src):
            os.symlink(os.path.join(src, f), os.path.join(train_dir, cls, f))

    # Val: fold_2
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    src = os.path.join(folds[-1], cls)  # fold_2
    for f in os.listdir(src):
        os.symlink(os.path.join(src, f), os.path.join(val_dir, cls, f))

# Print counts
for split, d in [('Train', train_dir), ('Val', val_dir)]:
    for cls in ['all', 'hem']:
        print(f'{split}/{cls}: {len(os.listdir(os.path.join(d, cls)))} images')

## Data Preprocessing

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Only rescale, no augmentation
datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True
)

val_gen = datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

print('Classes:', train_gen.class_indices)

## Model Building

In [ ]:
# Load VGG16 without top layers, freeze it
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# Add classification head
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.summary()

## Model Compilation

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## Model Training

In [ ]:
EPOCHS = 10

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen
)

## Evaluation

In [ ]:
val_loss, val_acc = model.evaluate(val_gen)
print(f'Validation Accuracy: {val_acc*100:.2f}%')
print(f'Validation Loss: {val_loss:.4f}')

## Results Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Predictions
val_gen.reset()
y_pred = (model.predict(val_gen) > 0.5).astype(int).flatten()
y_true = val_gen.classes

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['ALL', 'HEM'], yticklabels=['ALL', 'HEM'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Classification Report
print(classification_report(y_true, y_pred, target_names=['ALL (Leukemia)', 'HEM (Healthy)']))